# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb 
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient


from lib.agents_state_machine import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from lib.parsers import PydanticOutputParser


In [3]:
load_dotenv()

#OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
#TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")


True

In [4]:
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in .env"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY not found in .env"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL not found in .env"
print("✓ Environment variables loaded successfully")

✓ Environment variables loaded successfully


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


### Evaluation Report Model

In [5]:
class EvaluationReport(BaseModel):
    """Evaluation report for retrieval quality"""
    useful: bool = Field(description="Whether the retrieved documents are useful to answer the question")
    description: str = Field(description="Detailed explanation of the evaluation and suggested next action")

print("✓ EvaluationReport model defined")

✓ EvaluationReport model defined


In [6]:
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_base=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name="text-embedding-3-small"
)

#### Retrieve Game Tool

In [7]:
@tool
def retrieve_game(query: str) -> str:
    """Semantic search: Finds most similar games in the vector DB.
    
    Args:
        query: A question about game industry.
    
    Returns:
        List of game information. Each element contains:
        - Platform: like Game Boy, PlayStation 5, Xbox 360
        - Name: Name of the Game
        - YearOfRelease: Year when that game was released for that platform
        - Description: Additional details about the game
        - Genre: Game genre
        - Publisher: Game publisher
    """
    try:
        chroma_client = chromadb.PersistentClient(path="chromadb")
        collection = chroma_client.get_or_create_collection(name="udaplay",embedding_function=embedding_fn)
        results = collection.query(
            query_texts=[query],
            n_results=3
        )
        if not results['metadatas'] or not results['metadatas'][0]:
            return json.dumps({"results": [], "message": "No games found"})
        
        games_info = []
        for metadata, distance in zip(results['metadatas'][0], results['distances'][0]):
            game_data = {
                "Name": metadata.get("Name", "Unknown"),
                "Platform": metadata.get("Platform", "Unknown"),
                "YearOfRelease": metadata.get("YearOfRelease", "Unknown"),
                "Genre": metadata.get("Genre", "Unknown"),
                "Publisher": metadata.get("Publisher", "Unknown"),
                "Description": metadata.get("Description", "No description"),
                "relevance_score": round(1 - distance, 3)
            }
            games_info.append(game_data)
        
        return json.dumps({"results": games_info, "count": len(games_info)})
    except Exception as e:
        return json.dumps({"error": str(e), "results": []})

#### Evaluate Retrieval Tool

In [8]:
# Creating evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> str:
    """Evaluates if retrieved documents are sufficient to answer the user's question.
    
    Based on the user's question and the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    
    Args:
        question: Original question from user
        retrieved_docs: Retrieved documents most similar to the user query in the Vector Database
    
    Returns:
        Evaluation report with:
        - useful: Whether the documents are useful to answer the question
        - description: Description about the evaluation result and recommended action
    """
    try:
        # Parse the retrieved docs if it's a JSON string
        if isinstance(retrieved_docs, str):
            docs_data = json.loads(retrieved_docs)
            docs_list = docs_data.get('results', [])
        else:
            docs_list = retrieved_docs
        
        # Use LLM as judge with custom base URL
        evaluator_llm = LLM(
            model="gpt-4o-mini", 
            temperature=0
        )
        
        prompt = f"""Your task is to evaluate if the retrieved documents are sufficient to answer the user's question.

Question: {question}

Retrieved Documents:
{json.dumps(docs_list, indent=2)}

Analyze:
1. Do the documents contain relevant information to answer the question?
2. Is the information complete enough?
3. What is the quality of the match?

Provide a detailed evaluation explaining whether these documents are useful and what action should be taken.
"""
        
        response = evaluator_llm.invoke(
            input=prompt,
            response_format=EvaluationReport
        )
        
        # Parse the response
        parser = PydanticOutputParser(model_class=EvaluationReport)
        evaluation = parser.parse(response)
        
        return json.dumps({
            "useful": evaluation.useful,
            "description": evaluation.description
        })
    
    except Exception as e:
        # Fallback evaluation based on simple heuristics
        if not docs_list or len(docs_list) == 0:
            return json.dumps({
                "useful": False,
                "description": "No documents were retrieved. Consider using web search for this query."
            })
        
        # Check relevance scores if available
        avg_score = sum(d.get('relevance_score', 0) for d in docs_list) / len(docs_list) if docs_list else 0
        
        if avg_score < 0.5:
            return json.dumps({
                "useful": False,
                "description": f"Low relevance scores (avg: {avg_score:.2f}). The retrieved documents may not be relevant. Consider web search."
            })
        
        return json.dumps({
            "useful": True,
            "description": f"Retrieved {len(docs_list)} documents with average relevance score {avg_score:.2f}. Should be sufficient to answer."
        })

#### Game Web Search Tool

In [9]:
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def game_web_search(question: str) -> str:
    """Search the web for information about video games when internal knowledge is insufficient.
    
    Args:
        question: A question about game industry
    
    Returns:
        Web search results with relevant information
    """
    try:
        # Use Tavily for web search
        response = tavily_client.search(
            query=question,
            search_depth="basic",
            max_results=3
        )
        
        results = []
        for result in response.get('results', []):
            results.append({
                "title": result.get('title', ''),
                "content": result.get('content', ''),
                "url": result.get('url', ''),
                "score": result.get('score', 0)
            })
        
        return json.dumps({
            "results": results,
            "count": len(results),
            "answer": response.get('answer', '')
        })
    
    except Exception as e:
        return json.dumps({
            "error": str(e),
            "results": []
        })


### Agent

In [10]:
# Create the UdaPlay Agent
instructions = "You are an AI agent that can give insights about a game based on user's questions"

agent_state_machine = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.7
)

In [11]:
query1 = "When was Pokémon Gold and Silver released?"
print(f"Query: {query1}\n")
print("=" * 80)

run1 = agent_state_machine.invoke(query1)
final_state1 = run1.get_final_state()

# Get the final AI response
for msg in reversed(final_state1['messages']):
    if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
        print("\nAgent Response:")
        print("-" * 80)
        print(msg.content)
        break

print("\n" + "=" * 80)
print(f"Total tokens used: {final_state1.get('total_tokens', 0)}")

Query: When was Pokémon Gold and Silver released?

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: recall_memory
[StateMachine] Executing step: game_retrieval
[StateMachine] Executing step: answer_evaluator
[StateMachine] Executing step: final_answer
[StateMachine] Terminating: __termination__

Agent Response:
--------------------------------------------------------------------------------
Pokémon Gold and Silver were released in 1999 for the Game Boy Color. This information is sourced from the internal database.

Total tokens used: 609


In [12]:
final_state1

{'user_query': 'When was Pokémon Gold and Silver released?',
 'instructions': "You are an AI agent that can give insights about a game based on user's questions",
 'messages': [SystemMessage(role='system', content="You are an AI agent that can give insights about a game based on user's questions"),
  UserMessage(role='user', content='When was Pokémon Gold and Silver released?'),
  AIMessage(role='assistant', content='Pokémon Gold and Silver were released in 1999 for the Game Boy Color. This information is sourced from the internal database.', tool_calls=None, token_usage=None)],
 'evaluation': True,
 'evaluation_result': '{"useful": true, "description": "The retrieved documents contain relevant information to answer the user\'s question about the release date of Pok\\u00e9mon Gold and Silver. Specifically, the first document explicitly states that Pok\\u00e9mon Gold and Silver were released in 1999, which directly answers the question. \\n\\nThe second and third documents, while they p

In [13]:
query2 = "Which one was the first 3D platformer Mario game?"
print(f"Query: {query2}\n")
print("=" * 80)

run2 = agent_state_machine.invoke(query2)
final_state2 = run2.get_final_state()

# Get the final AI response
for msg in reversed(final_state2['messages']):
    if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
        print("\nAgent Response:")
        print("-" * 80)
        print(msg.content)
        break

print("\n" + "=" * 80)
print(f"Total tokens used: {final_state2.get('total_tokens', 0)}")

Query: Which one was the first 3D platformer Mario game?

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: recall_memory
[StateMachine] Executing step: game_retrieval
[StateMachine] Executing step: answer_evaluator
[StateMachine] Executing step: final_answer
[StateMachine] Terminating: __termination__

Agent Response:
--------------------------------------------------------------------------------
The first 3D platformer Mario game is **Super Mario 64**, which was released in 1996 for the Nintendo 64. This groundbreaking game set new standards for the genre and features Mario's quest to rescue Princess Peach. It marked a significant evolution in platforming games by introducing 3D environments, allowing players to explore and interact with the game world in ways that were not possible in earlier 2D platformers.

Total tokens used: 704


In [11]:
query3 = "Was Mortal Kombat X released for PlayStation 5?"
print(f"Query: {query3}\n")
print("=" * 80)

run3 = agent_state_machine.invoke(query3)
final_state3 = run3.get_final_state()

# Get the final AI response
for msg in reversed(final_state3['messages']):
    if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
        print("\nAgent Response:")
        print("-" * 80)
        print(msg.content)
        break

print("\n" + "=" * 80)
print(f"Total tokens used: {final_state3.get('total_tokens', 0)}")

Query: Was Mortal Kombat X released for PlayStation 5?

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: recall_memory
[StateMachine] Executing step: game_retrieval
[StateMachine] Executing step: answer_evaluator
[StateMachine] Executing step: web_search
[StateMachine] Executing step: persist_memory
[StateMachine] Executing step: final_answer
[StateMachine] Terminating: __termination__

Agent Response:
--------------------------------------------------------------------------------
Mortal Kombat X was not originally released for PlayStation 5. It was released for PlayStation 4, Xbox One, and PC. However, it is playable on PlayStation 5 through backward compatibility, though some features from the PlayStation 4 version may be absent. You can find more details on the PlayStation website.

For further confirmation, you can check the following sources:
1. [Mortal Kombat X - PlayStation](https://www.playstation.com/en-us/games/mor

In [12]:
query4 = "Was Mortal Kombat X released for PlayStation 5?"
print(f"Query: {query3}\n")
print("=" * 80)

run4 = agent_state_machine.invoke(query3)
final_state4 = run4.get_final_state()

# Get the final AI response
for msg in reversed(final_state3['messages']):
    if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
        print("\nAgent Response:")
        print("-" * 80)
        print(msg.content)
        break

print("\n" + "=" * 80)
print(f"Total tokens used: {final_state4.get('total_tokens', 0)}")

Query: Was Mortal Kombat X released for PlayStation 5?

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: recall_memory
[StateMachine] Executing step: final_answer
[StateMachine] Terminating: __termination__

Agent Response:
--------------------------------------------------------------------------------
Mortal Kombat X was not originally released for PlayStation 5. It was released for PlayStation 4, Xbox One, and PC. However, it is playable on PlayStation 5 through backward compatibility, though some features from the PlayStation 4 version may be absent. You can find more details on the PlayStation website.

For further confirmation, you can check the following sources:
1. [Mortal Kombat X - PlayStation](https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved)
2. [Mortal Kombat X | Warner Bros. Games - GameStop](https://www.gamestop.com/video-games/products/mortal-kombat-x/11003580.html)

Total tokens used: 518
